# Global66 — VoC Intelligence (Google Colab)

**Autor:** Vicente Muster · Mayo 2026

Este cuaderno **clona el repositorio** en el runtime de Colab y ejecuta el pipeline sobre una muestra del dataset. Sirve como alternativa cuando no quiere instalar Python en su PC.

Verá, entre otras cosas: batch de prueba (30 casos por defecto), `cost_report.json`, ejemplos de casos escalados y una demostración del flujo de análisis.

**Necesita:** una clave de API del proveedor que configure en `.env` (el ejemplo usa Gemini; puede adaptar el cuaderno a otro proveedor siguiendo `.env.example`).

Costes y límites dependen **solo** del plan y la consola del proveedor que use.

**Guía paso a paso en local o detalles de Postman:** en el repo, `Entregables/code_source/GUIA_EVALUADOR.md`.

---

### Contenido
1. Clonar el repo.
2. Instalar dependencias.
3. Configurar API key (sin imprimirla).
4. Smoke test (30 casos).
5. `cost_report.json` y muestra de escalados.
6. Demostración del analizador en proceso (sin servidor HTTP permanente en Colab free).
7. (Opcional) Ampliar el batch.

## 1. Clonar el repo

Este cuaderno clona el repo público de entrega. La URL por defecto es la del caso Global66; si cambias el nombre del repo en GitHub, edita `REPO_URL` en la siguiente celda.

In [ ]:
# Repo público entregado (HTTPS desde GitHub → Code)
REPO_URL = 'https://github.com/vmuster/global66-businesscase.git'
REPO_DIR = 'global66-businesscase'

import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('CWD:', os.getcwd())
!ls -la

## 2. Instalar dependencias

Toma ~60 segundos.

In [ ]:
!pip install -q -r requirements.txt

## 3. Pegar tu API key

La key NO se loguea ni se guarda en el notebook (uso `getpass`). Solo va al archivo `.env` local del runtime de Colab.

Si todavía no tienes una:
1. Ve a https://aistudio.google.com/
2. "Get API key" → "Create API key in new project"
3. Copia la key (empieza con `AIza...`)

In [ ]:
from getpass import getpass
import shutil, pathlib

key = getpass('Pega tu GEMINI_API_KEY (no se mostrará): ')
shutil.copy('.env.example', '.env')
env_path = pathlib.Path('.env')
content = env_path.read_text()
content = content.replace('GEMINI_API_KEY=', f'GEMINI_API_KEY={key}')
env_path.write_text(content)
print('Key configurada. Defaults free-tier seguros (Gemini 8 RPM, concurrency 1).')

## 4. Smoke test — 30 casos

Procesa los primeros 30 casos del dataset histórico. En free tier (8 RPM) toma ~4 minutos.

Con paid tier (recomendado para iteración) puedes ejecutar:
```bash
!python scripts/process_batch.py --limit 30 --rate-limit-rpm 60 --concurrency 4
```
Termina en ~30 segundos y cuesta ~$0.01 USD.

In [ ]:
!python scripts/process_batch.py --limit 30

## 5. Cost report y top casos escalados

Después del run, `data/cost_report.json` tiene los tokens reales y la proyección a 100k/mes.
El audit completo (`results_audit.json`) trae cada caso con su priority breakdown.

In [ ]:
import json
from pathlib import Path

cost = json.loads(Path('data/cost_report.json').read_text())
print('=== COST REPORT ===')
for k in ['messages_processed','tokens_in_total','tokens_out_total','cost_total_usd','cost_per_message_usd','projection_100k_monthly_usd','p50_latency_ms','p95_latency_ms','duration_seconds']:
    print(f'  {k:35s} {cost.get(k)}')

In [ ]:
audit = json.loads(Path('data/results_audit.json').read_text())
results = audit.get('results', audit)
escalated = [r for r in results if r.get('escalated')]
print(f'Casos escalados: {len(escalated)} / {len(results)}')
print('\n=== TOP 5 ESCALADOS (por score_final) ===')
top5 = sorted(escalated, key=lambda r: -(r.get('score', {}) or {}).get('score_final', 0))[:5]
for r in top5:
    s = r.get('score', {}) or {}
    a = r.get('analysis', {}) or {}
    esc = (a.get('escalation') or {}) if isinstance(a, dict) else {}
    print(f"  case={r['case_id']:12s} priority={s.get('priority_final')}  score={s.get('score_final')}  team={esc.get('suggested_human_team')}")
    print(f"    reason: {esc.get('reason')}")

## 6. Webhook en vivo (sin levantar uvicorn)

Llamamos al orchestrator directamente desde Python con un caso sintético, demostrando que el sistema responde **en menos de 2 segundos** con un análisis estructurado completo.

In [ ]:
import asyncio, time
import sys
sys.path.insert(0, '.')

from src.core.engine import build_client_from_env
from src.core.orchestrator import analyze_and_persist, ingest_message
from src.core.schema import WebhookPayload
from src.database import db
from dotenv import load_dotenv
load_dotenv()
db.init_db()

client = build_client_from_env()
print(f'Active LLM: {client.provider_name} / {client.model}')

payload = WebhookPayload(
    case_id='COLAB-DEMO-001',
    message_id='COLAB-DEMO-MSG-1',
    user_id='colab-user',
    direction='INBOUND',
    text='Mi cuenta fue hackeada, dice que entraron desde Rusia y vaciaron $5000 USD. Bloqueen todo, voy a la CMF.',
    pais_usuario='Chile',
    platform='whatsapp',
)

async def run():
    t0 = time.perf_counter()
    meta = ingest_message(payload, synthetic_ts=False)
    res = await analyze_and_persist(
        client=client,
        case_id=payload.case_id,
        message_id=payload.message_id,
        user_pseudonym=meta['user_pseudonym'],
        country_iso=meta['country_iso'],
        detected_language=meta['language'],
    )
    dt = (time.perf_counter() - t0) * 1000
    print(f'\n=== Webhook response ({dt:.0f} ms) ===')
    print(f'  status              {res["status"]}')
    print(f'  escalated           {res["escalated"]}')
    print(f'  priority_final      {res["score"]["priority_final"]} (math={res["score"]["priority_math"]}, llm={res["score"]["priority_llm"]})')
    print(f'  suggested_team      {res["analysis"].escalation.suggested_human_team}')
    print(f'  reason              {res["analysis"].escalation.reason}')
    print(f'  regulatory_flags    {[(f.type, f.jurisdiction) for f in res["analysis"].regulatory_flags]}')
    print(f'  confidence          {res["analysis"].confidence}')
    print(f'  tokens              {res["tokens"]}  model={res["model_used"]}')

asyncio.run(run())

## 7. (Opcional) Dataset completo — 250 casos

Si quieres ver el batch completo, descomenta la celda. En **free tier** toma ~30 minutos (Gemini 8 RPM).  En **paid tier** con el tunning recomendado (`--rate-limit-rpm 200 --concurrency 8`) toma ~3 minutos y cuesta ~$0.08 USD.

El sistema hace **resume** desde la DB: si interrumpes y vuelves a ejecutar el batch, no re-procesa los mensajes ya analizados.

In [ ]:
# Free tier (paciencia, ~30 min):
# !python scripts/process_batch.py

# Paid tier agresivo (con billing activado en aistudio.google.com):
# !python scripts/process_batch.py --rate-limit-rpm 200 --concurrency 8 --no-resume

## 8. Trazabilidad y dashboard local

Lo siguiente requiere clonar el repo localmente (no funciona en Colab):
- Levantar el dashboard Streamlit (`streamlit run src/dashboard/app.py`).
- Probar con Postman / Insomnia (necesitan red local hacia `uvicorn`).

Ambas rutas están documentadas en `Entregables/README_ENTREGA.md` (Ruta B).

---

**Lo que viste en este notebook:**
- Sistema multilingüe (es/pt/en/fr) detecta fraude y compliance en menos de 2 segundos por caso.
- Costo proyectado a 100k mensajes/mes: ~$32 USD/mes con Gemini paid.
- Schema Pydantic estricto: la respuesta JSON **siempre** es válida o reintenta.
- Priority breakdown auditable: ves cómo se llegó a la decisión final (math + LLM + override anti-memoria).
- Cero alucinación en payloads de prueba: el input guard intercepta `text='string'` antes del LLM.